# 作业 03：在 PyTorch 中调用交叉熵

本次只判断一件事：**有没有行人？** 不使用图片、CNN 或数据下载。我们用一个简单函数产生两个预测概率，把注意力放在最后的损失函数调用上。

类别的顺序始终固定：

| 类别编号 | 含义 | one-hot 标签 y |
|---|---|---|
| 0 | 没有行人 | `[1, 0]` |
| 1 | 有行人 | `[0, 1]` |

`y` 和 `y_pred` 都表示**同一个样本的两个类别**，shape 为 `(2,)`。`y_pred[0]` 是没有行人的概率，`y_pred[1]` 是有行人的概率，两者相加为 1。

目标：会调用 `F.cross_entropy`，并能解释概率变大或变小时，损失为什么变化。


In [ ]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)

def show(name, value):
    print(f'{name}: {value}, shape={tuple(value.shape)}, dtype={value.dtype}')

def predict(x):
    return torch.stack([1 - x, x])

# 修改 x 为 0.8
x = torch.tensor(0.8)
y = torch.tensor([1., 0.])  # 真实情况：没有行人 (类别 0)
y_pred = predict(x)

show('x', x)
show('y (one-hot 真实标签)', y)
show('y_pred (两个预测概率)', y_pred)
show('预测概率之和', y_pred.sum())

# 计算损失
scores = torch.log(y_pred)
label = y.argmax()  # 真实类别编号 0
loss = F.cross_entropy(scores, label)
show('loss', loss)

## 1. 一个输出两个概率的预测函数

为了便于观察，直接把输入 x 当作“有行人的预测概率”，于是“没有行人的概率”就是 `1-x`。这是人为设置的教学函数，不是训练好的行人识别器。

先取 x=0.2，则预测应为 `[0.8, 0.2]`。


In [ ]:
def predict(x):
    # x 是一个 0 到 1 之间的浮点标量张量
    return torch.stack([1 - x, x])

x = torch.tensor(0.2)
y = torch.tensor([1., 0.])  # 真实情况：没有行人
y_pred = predict(x)

show('x', x)
show('y（one-hot 真实标签）', y)
show('y_pred（两个预测概率）', y_pred)
show('预测概率之和', y_pred.sum())


## 2. 最后调用 cross_entropy 计算损失

**PyTorch 的 `F.cross_entropy` 接收原始分数 logits，而不是直接接收概率。**

本例的函数已经输出概率，因此先用 `torch.log(y_pred)` 构造一组等价分数：对这组分数做 softmax，会还原原来的概率。这里两个概率均大于 0。

真实标签使用类别编号：`y=[1,0]` 对应编号 `0`，`y=[0,1]` 对应编号 `1`。用 `y.argmax()` 取出这个编号，得到整数标量。这里取的是**真实标签**的编号，不是预测的 argmax。

请重点看最后一行：损失由 `F.cross_entropy(...)` 计算。


In [ ]:
scores = torch.log(y_pred)  # 从本例的概率构造等价 logits
label = y.argmax()        # [1, 0] → 类别编号 0，dtype 为 torch.int64

show('scores（传给损失函数的分数）', scores)
show('softmax(scores)（还原概率）', scores.softmax(dim=0))
show('label（真实类别编号）', label)

loss = F.cross_entropy(scores, label)
show('loss', loss)


### 对照手算

展开公式：

$$L=-[y_0\ln(p_0)+y_1\ln(p_1)].$$

真实标签为 `[1,0]` 时，只留下 $-\ln(p_0)$。当前 $p_0=0.8$，所以损失约为 **0.2231**。

下面仅用于核对；实际调用损失函数时，仍使用上面的 `F.cross_entropy`。


In [ ]:
manual_loss = -torch.log(y_pred[label])
show('手算：真实类别概率的负对数', manual_loss)
print('与 cross_entropy 一致：', torch.allclose(loss, manual_loss))


## 3. 几个带数字的例子

下面固定真实标签为 `[1,0]`，即“没有行人”。只改变模型的预测概率。

| 真实标签 y | 预测概率 y_pred | 交叉熵 |
|---|---|---|
| `[1,0]` | `[0.9,0.1]` | $-\ln(0.9)\approx0.1054$ |
| `[1,0]` | `[0.5,0.5]` | $-\ln(0.5)\approx0.6931$ |
| `[1,0]` | `[0.1,0.9]` | $-\ln(0.1)\approx2.3026$ |
| `[1,0]` | `[0,1]` | $-\ln(0)\to+\infty$：把真实类别的概率设为 0 |
| `[1,0]` | `[1,0]` | $-\ln(1)=0$：完全相信真实类别 |

概率越接近真实标签，损失越小。概率精确为 0 的情况是边界示例；PyTorch 会在下面的错误预测例子中打印 `inf`，表示正无穷。


In [ ]:
def compare(y, probabilities):
    label = y.argmax()
    # 仅为展示指定概率的交叉熵。log(0)=-inf，是这里刻意保留的边界情况。
    scores = torch.log(probabilities)
    loss = F.cross_entropy(scores, label)
    print(f'y={y.tolist()}, y_pred={probabilities.tolist()}')
    print(f'真实类别概率={probabilities[label].item():.4f}, loss={loss.item():.4f}')
    return loss

y = torch.tensor([1., 0.])
example_losses = []
for values in [[0.9, 0.1], [0.5, 0.5], [0.1, 0.9], [0., 1.], [1., 0.]]:
    example_losses.append(compare(y, torch.tensor(values)))
    print()


## 4. 换成“有行人”试试看

现在 y 为 `[0,1]`，真实类别是 1，应当关注第二个概率。下面预测为 `[0.1,0.9]`，损失约为 **0.1054**。同一个预测在上节被惩罚，在这里得到较小损失，是因为真实答案变了。


In [ ]:
y = torch.tensor([0., 1.])
y_pred = torch.tensor([0.1, 0.9])
compare(y, y_pred)


## 5. 真正的网络输出原始分数时，直接调用

前面的 `torch.log` 是为了配合本题“给定概率”的数值演示。真实网络通常直接输出两个原始分数，此时**直接把分数交给 cross_entropy**，不需要先 softmax 或取 log。

下面的 scores 为 `[2,-1]`，它们不是概率，也不需要加起来等于 1。softmax 只用来把概率打印给我们看，不是交叉熵的输入。


In [ ]:
scores = torch.tensor([2., -1.])
y = torch.tensor([1., 0.])
label = y.argmax()

show('原始分数 scores', scores)
show('给人看的预测概率', scores.softmax(dim=0))
loss = F.cross_entropy(scores, label)
show('直接调用 cross_entropy 得到的 loss', loss)


## 你的实验（50分）

1. 把第 1 节的 x 改成 `0.8`，从第 1 节重新运行到手算核对。保持真实标签为 `[1,0]`。记录 y_pred 和损失，解释为什么损失变大。
2. 在下面补全代码：真实标签是“有行人”，预测概率为 `[0.8,0.2]`。打印两个张量的 shape，再调用 cross_entropy，预期损失约为 `1.6094`。
3. 用自己的话解释：为什么 `[1,0]` 对 `[0,1]` 的损失是无穷大，而 `[1,0]` 对 `[1,0]` 的损失是 0？
4. `F.cross_entropy(y_pred, label)` 与 `F.cross_entropy(torch.log(y_pred), label)` 中，哪一个才是在计算这里指定的预测概率对应的交叉熵？原始网络输出 scores 时，又该怎样调用？


In [5]:
import torch
import torch.nn.functional as F

# 1. 定义真实标签
# “有行人”对应类别 1，one-hot 编码为 [0, 1]
y = torch.tensor([0., 1.]) 

# 2. 定义预测概率
# 模型预测类别 0 的概率为 0.8，类别 1 的概率为 0.2
y_pred = torch.tensor([0.8, 0.2])

# 3. 打印 shape
print(f"y shape: {y.shape}")      # 输出: torch.Size([2])
print(f"y_pred shape: {y_pred.shape}") # 输出: torch.Size([2])

# 4. 计算交叉熵损失
# 注意：F.cross_entropy 第一个参数需要是 log(概率) 或者原始 logits
# 这里 y_pred 已经是概率，所以传入 torch.log(y_pred)
label = y.argmax() # 获取类别索引，即 1
loss = F.cross_entropy(torch.log(y_pred), label.unsqueeze(0)) 

print(f"Loss: {loss.item():.4f}")

y shape: torch.Size([2])
y_pred shape: torch.Size([2])


RuntimeError: size mismatch (got input: [2], target: [1])

### 我的回答 (50分)

- 修改 x 后的预测与损失：解释:当 x = 0.2 时，y_pred = [0.8, 0.2]，真实类别 0 的概率是 0.8，损失为 -ln(0.8) approx 0.2231。当 x = 0.8 时，y_pred = [0.2, 0.8]，真实类别 0 的概率是 0.2，损失为 -ln(0.2) approx 1.6094。
损失变大的原因：真实标签是“没有行人”（类别 0），而模型预测“有行人”（类别 1）的概率变高了（0.8），即真实类别的概率从 0.8 降到了 0.2。交叉熵损失 -ln(p_0) 随着 p_0 减小而增大，说明模型预测与真实情况偏差更大，因此损失变大。

- “有行人”实验的结果：
- 两个极端例子的解释：交叉熵公式为 L = -sum y_i ln(p_i)。对于 one-hot 标签，只有真实类别的概率 p_{true} 起作用，即 L = -ln(p_{true})。
当真实标签为 [1, 0]（类别 0），预测为 [0, 1] 时，真实类别 0 的概率 p_0 = 0。-ln(0) to +infty，因为模型完全否定了真实情况，惩罚极大。
当预测为 [1, 0] 时，p_0 = 1。-ln(1) = 0，因为模型完全确信真实类别，损失为 0。

- 概率与原始分数应该怎样传入损失函数：PyTorch 的  F.cross_entropy  接收的是原始分数（logits），内部会先做  softmax  再算交叉熵。
如果 y_pred 是概率（如 [0.8, 0.2]），直接传入会把它当 logits。此时应传入  torch.log(y_pred)  作为 scores，因为 softmax(log(p)) = p（在概率和为 1 时等价）。
所以， F.cross_entropy(torch.log(y_pred), label)  是计算指定概率对应的交叉熵。
原始网络输出 scores（如 [2, -1]）时，直接调用  F.cross_entropy(scores, label)  即可，不需要再取 log。

### 提交

补全实验代码和文字回答，重启内核后运行全部单元格并保存输出。示例中的 `inf` 是刻意展示的无穷大，不是代码异常。本题不要求编写训练循环。

参考：[PyTorch CrossEntropyLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)。
